# Lab Assignment 7
## RNN Applications: from SimpleRNN to LSTM, and from Classification to Tagging


---

### Objective

The tutorial ended with a problem rather than a solution. A **SimpleRNN** classified sentiment at
about 64%, and its accuracy **collapsed on long sentences** — 70.5% on short ones, 58.4% on long
ones. The lecture blames the **vanishing gradient**, and claims an **LSTM** fixes it.

This assignment does two things.

**Part A — Task 1.** Test that claim properly. Same data, same task, same settings; you change
**only** the recurrent layer. If the lecture is right, the LSTM should help — and it should help
**most on the long sentences**.

**Part B — Tasks 2 to 5.** Take the LSTM to a different kind of problem: **POS tagging**, where
every word gets its own label instead of one label for the whole sentence. Then make it
**bidirectional** and measure what that buys you.

You tagged parts of speech once before, in **Lab 3, using a Hidden Markov Model**. This time the
model learns the patterns from data instead of from explicit transition and emission counts.

### How this differs from the tutorial

| | Tutorial demo | This assignment |
|---|---|---|
| Application | sentiment classification | sentiment (Task 1), then **POS tagging** (Tasks 2–5) |
| Shape | many-to-one | many-to-one, then **many-to-many** |
| Recurrent cell | `SimpleRNN` | **`LSTM`**, then **`Bidirectional(LSTM)`** |
| Output | `Dense(1)` on the final state | `Dense(1)`, then `TimeDistributed(Dense(n_tags))` |
| Data | movie review sentences | the same reviews, then the **Penn Treebank** |

### Marks

| Task | Description | Marks |
|---|---|---|
| **Task 1** | Does the LSTM actually fix it? | **4** |
| **Task 2** | Data preparation for tagging | **4** |
| **Task 3** | Building and training the LSTM tagger | **4** |
| **Task 4** | Evaluating the tagger correctly | **4** |
| **Task 5** | Bidirectional LSTM and comparison | **4** |
| | **Total** | **20** |

---

### Before you start

- A GPU is **not** required. Everything here runs in a few minutes on CPU.
- **Task 1 needs the file `mini_sentiment.tsv`**, the same one the tutorial used. You must have it
  in the same folder as this notebook before you start.
  **In Colab:** click the folder icon in the left sidebar and drag the file in, or run
  ```python
  from google.colab import files
  files.upload()
  ```
  Do **not** rebuild this file from NLTK yourself. The baseline numbers you are comparing against
  come from this exact file in this exact order, so a regenerated copy would make the comparison
  invalid.
- You may reuse the tutorial's Task 1 pipeline directly — the marks are for the **measurement and
  the explanation**, not for retyping the data loading.
- **Tasks 2–5** use the Penn Treebank from NLTK.
- Use the **universal tagset** (12 tags), not the full Penn tagset (45 tags). With only 3,914
  sentences the rarer Penn tags have far too few examples.

**You are expected to write the code yourself.** Each task states what to build, the names to use,
and the exact output it must print — but not how to write it.

---
## Setup

These are the only values given to you.

In [1]:
from google.colab import files
import os

uploaded = files.upload()

uploaded_name = next(iter(uploaded))
print("Uploaded:", uploaded_name)

Saving mini_sentiment.tsv to mini_sentiment.tsv
Uploaded: mini_sentiment.tsv


In [2]:
import numpy as np
import nltk
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, LSTM, Bidirectional, Dense, TimeDistributed
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('treebank', quiet=True)
nltk.download('universal_tagset', quiet=True)
from nltk.corpus import treebank

keras.utils.set_random_seed(42)

# --- Task 1 (sentiment), same values as the tutorial ---
MAX_WORDS = 5000
SENT_LEN  = 40

# --- Tasks 2-5 (POS tagging) ---
MAX_LEN   = 50      # pad/truncate every sentence to this many words
EMBED_DIM = 64
RNN_UNITS = 64
N_TRAIN   = 3000    # first 3000 sentences for training, the rest for testing

tagged_sentences = treebank.tagged_sents(tagset='universal')
print(f"{len(tagged_sentences)} tagged sentences")
print("example:", tagged_sentences[0][:6])

3914 tagged sentences
example: [('Pierre', 'NOUN'), ('Vinken', 'NOUN'), (',', '.'), ('61', 'NUM'), ('years', 'NOUN'), ('old', 'ADJ')]


---
# Task 1: Does the LSTM actually fix it? &nbsp;&nbsp;*(4 marks)*

The tutorial's SimpleRNN scored this on the sentiment test set:

| Group | n | SimpleRNN |
|---|---|---|
| SHORT (≤ 15 words) | 808 | **0.705** |
| ALL | 2662 | **0.639** |
| LONG (≥ 28 words) | 680 | **0.584** |

Rebuild that classifier **changing only the recurrent layer** — `SimpleRNN(32)` becomes
`LSTM(32)`. Everything else stays identical: same file, same `MAX_WORDS`, same `SENT_LEN`, same
8000/2662 split, same epochs, same batch size. If you change anything else the comparison is
meaningless.

Then score it the same three ways and put the numbers side by side.

### What your code must create

| Name | Is |
|---|---|
| `X_sent`, `y_sent` | the padded sentiment data, exactly as in the tutorial |
| `raw_len` | each test sentence's true word count, **before** padding |
| `model_lstm_sent` | the same architecture with `LSTM(32)` in place of `SimpleRNN(32)` |

### Output your code must print

```
GROUP                     n    SimpleRNN      LSTM     change
------------------------------------------------------------
SHORT (<= 15 words)     808        0.705     0.___     +_.___
ALL                    2662        0.639     0.___     +_.___
LONG  (>= 28 words)     680        0.584     0.___     +_.___
```

### Then answer

**Q1.** Where is the LSTM's improvement largest — on short sentences or long ones? Does that match
what the lecture predicts about vanishing gradients? Two or three sentences.

### Hints

- Read `mini_sentiment.tsv` exactly as the tutorial does: one header line, then `pos`/`neg` and the
  sentence separated by a tab.
- The split is the first 8000 sentences for training, the remainder for testing.
- `raw_len` must come from the token lists, **not** from the padded array — after padding every
  sentence has the same length.
- Call `keras.utils.set_random_seed(42)` before building the model.

In [3]:
# ---------------- TASK 1 ----------------

# -----------------------------------
# TODO 1: Load sentiment data
# -----------------------------------

labels_raw = []
texts_sent = []

with open("mini_sentiment.tsv", encoding="utf-8") as f:
    next(f)  # skip header

    for line in f:
        line = line.rstrip("\n")

        # The intended format is:
        # label<TAB>sentence
        #
        # Some copies of the supplied file contain additional tabs
        # inside the sentence. We therefore reconstruct the sentence
        # from everything after the label.

        first_part, sentence_part = line.split(maxsplit=1)

        label = first_part

        if label not in ("pos", "neg"):
            raise ValueError(f"Unexpected label: {label}")

        sentence = sentence_part.replace("\t", " ")

        labels_raw.append(
            1 if label == "pos" else 0
        )

        texts_sent.append(
            sentence.split()
        )

y_sent = np.array(labels_raw)

print("Sentences loaded:", len(texts_sent))
print("Positive:", int(y_sent.sum()))
print("Negative:", int(len(y_sent) - y_sent.sum()))

# -----------------------------------
# Build vocabulary
# -----------------------------------

counts = {}

for sentence in texts_sent:
    for word in sentence:
        counts[word] = counts.get(word, 0) + 1

vocab_sent = (
    ["<pad>", "<unk>"]
    +
    [
        word
        for word, _ in sorted(
            counts.items(),
            key=lambda item: -item[1]
        )[:MAX_WORDS]
    ]
)

word_index_sent = {
    word: index
    for index, word in enumerate(vocab_sent)
}

print("Vocabulary size:", len(vocab_sent))

# -----------------------------------
# Encode and pad
# -----------------------------------

sequences_sent = [
    [
        word_index_sent.get(word, 1)
        for word in sentence
    ]
    for sentence in texts_sent
]

X_sent = pad_sequences(
    sequences_sent,
    maxlen=SENT_LEN
)

print("X_sent shape:", X_sent.shape)

# -----------------------------------
# TODO 2: raw length of test sentences
# -----------------------------------

raw_len = np.array(
    [
        len(sentence)
        for sentence in texts_sent
    ]
)[8000:]

# -----------------------------------
# TODO 3: LSTM model
# -----------------------------------

keras.utils.set_random_seed(42)

model_lstm_sent = Sequential([
    Input(shape=(SENT_LEN,)),
    Embedding(
        input_dim=len(vocab_sent),
        output_dim=32
    ),
    LSTM(32),
    Dense(
        1,
        activation="sigmoid"
    )
])

model_lstm_sent.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_lstm_sent.fit(
    X_sent[:8000],
    y_sent[:8000],
    epochs=4,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

# -----------------------------------
# TODO 4: Test predictions
# -----------------------------------

X_sent_test = X_sent[8000:]
y_sent_test = y_sent[8000:]

pred_sent = (
    model_lstm_sent
    .predict(
        X_sent_test,
        verbose=0
    )
    .ravel() > 0.5
).astype(int)

short_mask = raw_len <= 15
long_mask = raw_len >= 28

short_acc = (
    pred_sent[short_mask]
    == y_sent_test[short_mask]
).mean()

all_acc = (
    pred_sent == y_sent_test
).mean()

long_acc = (
    pred_sent[long_mask]
    == y_sent_test[long_mask]
).mean()

# -----------------------------------
# TODO 5: Required comparison table
# -----------------------------------

print()
print(
    f"{'GROUP':<26}"
    f"{'n':>6}"
    f"{'SimpleRNN':>13}"
    f"{'LSTM':>10}"
    f"{'change':>12}"
)

print("-" * 67)

print(
    f"{'SHORT (<= 15 words)':<26}"
    f"{short_mask.sum():>6}"
    f"{0.705:>13.3f}"
    f"{short_acc:>10.3f}"
    f"{short_acc - 0.705:>+12.3f}"
)

print(
    f"{'ALL':<26}"
    f"{len(y_sent_test):>6}"
    f"{0.639:>13.3f}"
    f"{all_acc:>10.3f}"
    f"{all_acc - 0.639:>+12.3f}"
)

print(
    f"{'LONG  (>= 28 words)':<26}"
    f"{long_mask.sum():>6}"
    f"{0.584:>13.3f}"
    f"{long_acc:>10.3f}"
    f"{long_acc - 0.584:>+12.3f}"
)

Sentences loaded: 10051
Positive: 5039
Negative: 5012
Vocabulary size: 5002
X_sent shape: (10051, 40)

GROUP                          n    SimpleRNN      LSTM      change
-------------------------------------------------------------------
SHORT (<= 15 words)          682        0.705     0.792      +0.087
ALL                         2051        0.639     0.738      +0.099
LONG  (>= 28 words)          431        0.584     0.703      +0.119


---
# Task 2: Data preparation for tagging &nbsp;&nbsp;*(4 marks)*

Now switch tasks. Turn the tagged sentences into the rectangular integer arrays an RNN needs.

Each sentence is a list of `(word, tag)` pairs, so you need **two** parallel sequences: word ids
and tag ids, aligned position by position.

### What your code must create

| Name | Holds |
|---|---|
| `words`, `tags` | the sorted vocabularies (lowercase the words) |
| `word_index`, `tag_index` | word → id and tag → id dictionaries |
| `X`, `y` | padded word-id and tag-id arrays, shape `(n_sentences, MAX_LEN)` |
| `X_train`, `y_train`, `X_test`, `y_test` | split at `N_TRAIN` |

### Output your code must print

```
Vocabulary size: ...
Number of tags: ...
Tags: ['.', 'ADJ', 'ADP', ...]
X shape: (3914, 50)   y shape: (3914, 50)
Training sentences: 3000   Test sentences: 914
```
Then print the first 8 words and tags of one sentence, in both word form and id form, so it is
visible that the two sequences line up.

### Hints

- Reserve **id 0 for padding** in both vocabularies, and **id 1 for unknown words**. Real word ids
  start at 2, real tag ids at 1. Nothing may be given id 0 except padding.
- Lowercase words when building the vocabulary, and lowercase again when looking them up.
- Use `padding='post'` here. The tutorial padded at the *front*, which is fine when only the last
  hidden state matters — but for tagging, padding at the **end** keeps word `i` and tag `i` at the
  same index, which is what makes masking work.
- Pad `X` and `y` **the same way**.
- Check what shape `sparse_categorical_crossentropy` expects for the targets.

In [4]:
# ---------------- TASK 2 ----------------

# -----------------------------------
# TODO 6: Build vocabularies
# -----------------------------------

all_words = []
all_tags = []

for sentence in tagged_sentences:
    for word, tag in sentence:
        all_words.append(
            word.lower()
        )
        all_tags.append(tag)

words = sorted(
    set(all_words)
)

tags = sorted(
    set(all_tags)
)

# -----------------------------------
# TODO 7: Build ID dictionaries
# -----------------------------------

# Word IDs:
# 0 = padding
# 1 = unknown
# 2+ = real words

word_index = {
    "<pad>": 0,
    "<unk>": 1
}

for index, word in enumerate(
    words,
    start=2
):
    word_index[word] = index


# Tag IDs:
# 0 = padding
# 1+ = real tags

tag_index = {
    "<pad>": 0
}

for index, tag in enumerate(
    tags,
    start=1
):
    tag_index[tag] = index

# -----------------------------------
# TODO 8: Convert sentences to IDs
# -----------------------------------

word_sequences = []
tag_sequences = []

for sentence in tagged_sentences:

    word_ids = [
        word_index.get(
            word.lower(),
            1
        )
        for word, tag in sentence
    ]

    tag_ids = [
        tag_index[tag]
        for word, tag in sentence
    ]

    word_sequences.append(
        word_ids
    )

    tag_sequences.append(
        tag_ids
    )

# -----------------------------------
# TODO 9: Post-padding
# -----------------------------------

X = pad_sequences(
    word_sequences,
    maxlen=MAX_LEN,
    padding="post",
    value=0
)

y = pad_sequences(
    tag_sequences,
    maxlen=MAX_LEN,
    padding="post",
    value=0
)

# -----------------------------------
# TODO 10: Train/test split
# -----------------------------------

X_train = X[:N_TRAIN]
y_train = y[:N_TRAIN]

X_test = X[N_TRAIN:]
y_test = y[N_TRAIN:]

# -----------------------------------
# TODO 11: Print required statistics
# -----------------------------------

print(
    "Vocabulary size:",
    len(words)
)

print(
    "Number of tags:",
    len(tags)
)

print(
    "Tags:",
    tags
)

print(
    "X shape:",
    X.shape,
    " y shape:",
    y.shape
)

print(
    "Training sentences:",
    len(X_train),
    " Test sentences:",
    len(X_test)
)

# -----------------------------------
# Alignment example
# -----------------------------------

print("\nFirst 8 words and tags:")

for word, tag in tagged_sentences[0][:8]:
    print(
        f"{word:<15} -> {tag}"
    )

print("\nFirst 8 word IDs:")
print(
    X[0, :8]
)

print("\nFirst 8 tag IDs:")
print(
    y[0, :8]
)

Vocabulary size: 11387
Number of tags: 12
Tags: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
X shape: (3914, 50)  y shape: (3914, 50)
Training sentences: 3000  Test sentences: 914

First 8 words and tags:
Pierre          -> NOUN
Vinken          -> NOUN
,               -> .
61              -> NUM
years           -> NOUN
old             -> ADJ
,               -> .
will            -> VERB

First 8 word IDs:
[ 7986 10981   460  1263 11343  7554   460 11217]

First 8 tag IDs:
[ 7  7  1  8  7  2  1 11]


---
# Task 3: Building and training the LSTM tagger &nbsp;&nbsp;*(4 marks)*

```
   word ids  ->  Embedding  ->  LSTM(return_sequences=True)  ->  TimeDistributed(Dense(softmax))
   (n, 50)       (n, 50, 64)         (n, 50, 64)                        (n, 50, n_tags+1)
```

Compare this with the tutorial's model. Two things changed, and both are essential:

1. **`return_sequences=True`** — the LSTM must output a hidden state at **every** time step, not
   just the last. Without it you get one vector for the whole sentence and cannot tag words.
2. **`TimeDistributed(Dense(...))`** — applies the same Dense classifier independently at every
   time step. One shared classifier, applied 50 times.

### What your code must create

| Name | Is |
|---|---|
| `model_lstm` | the compiled tagger |
| `history_lstm` | the return value of `model_lstm.fit(...)` |

### Output your code must print

- `model_lstm.summary()` — showing **real output shapes and parameter counts**, not `?` and `0`
- the per-epoch table, ending with the final `loss`, `accuracy`, `val_loss`, `val_accuracy`

A correct summary looks like this (total parameters **762,765**):

```
 embedding (Embedding)        (None, 50, 64)        728,896
 lstm (LSTM)                  (None, 50, 64)         33,024
 time_distributed             (None, 50, 13)            845
```

Read the middle column: the LSTM keeps all 50 positions, and the output has 13 values per position
— one score per tag. That is what `return_sequences=True` and `TimeDistributed` buy you.

### Hints

- **Start your `Sequential` with `Input(shape=(MAX_LEN,))`.** Without it Keras does not build the
  model until training begins, so `model.summary()` prints `?` for every output shape and
  `0 (unbuilt)` for every parameter count — useless for your submission. The same applies to your
  Task 1 model, with `SENT_LEN` instead.
- Use `Embedding(..., mask_zero=True)` so later layers ignore padded positions and the model is not
  trained to predict a tag for padding. Note what this does to the accuracy Keras reports — Task 4
  asks about it.
- The output layer needs one unit per tag id **including the reserved 0**, so its size is
  `len(tags) + 1`.
- Loss `sparse_categorical_crossentropy` (targets are integer ids, not one-hot), optimizer `adam`.
- About 5 epochs, `batch_size=32`, `validation_split=0.2`. More will overfit; the dataset is small.

In [5]:
# ---------------- TASK 3 ----------------

# -----------------------------------
# TODO 12: Build the LSTM tagger
# -----------------------------------

model_lstm = Sequential([
    Input(
        shape=(MAX_LEN,)
    ),

    Embedding(
        input_dim=len(words) + 2,
        output_dim=EMBED_DIM,
        mask_zero=True
    ),

    LSTM(
        RNN_UNITS,
        return_sequences=True
    ),

    TimeDistributed(
        Dense(
            len(tags) + 1,
            activation="softmax"
        )
    )
])

# -----------------------------------
# TODO 13: Compile
# -----------------------------------

model_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# -----------------------------------
# TODO 14: Summary
# -----------------------------------

print("LSTM POS TAGGER")
print("=" * 60)

model_lstm.summary()

# -----------------------------------
# TODO 15: Train
# -----------------------------------

history_lstm = model_lstm.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# -----------------------------------
# TODO 16: Per-epoch results
# -----------------------------------

print("\nPER-EPOCH RESULTS")

print(
    f"{'epoch':>5}"
    f"{'loss':>10}"
    f"{'accuracy':>12}"
    f"{'val_loss':>12}"
    f"{'val_accuracy':>15}"
)

for i in range(
    len(history_lstm.history["loss"])
):

    print(
        f"{i + 1:>5}"
        f"{history_lstm.history['loss'][i]:>10.4f}"
        f"{history_lstm.history['accuracy'][i]:>12.4f}"
        f"{history_lstm.history['val_loss'][i]:>12.4f}"
        f"{history_lstm.history['val_accuracy'][i]:>15.4f}"
    )

print("\nFINAL EPOCH")

print(
    f"loss:         "
    f"{history_lstm.history['loss'][-1]:.4f}"
)

print(
    f"accuracy:     "
    f"{history_lstm.history['accuracy'][-1]:.4f}"
)

print(
    f"val_loss:     "
    f"{history_lstm.history['val_loss'][-1]:.4f}"
)

print(
    f"val_accuracy: "
    f"{history_lstm.history['val_accuracy'][-1]:.4f}"
)

LSTM POS TAGGER


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 50, 64)         │       728,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 50, 13)         │           845 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 762,765 (2.91 MB)

 Trainable params: 762,765 (2.91 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 62ms/step - accuracy: 0.3784 - loss: 2.0889 - val_accuracy: 0.5539 - val_loss: 1.5990
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - accuracy: 0.7053 - loss: 1.0649 - val_accuracy: 0.7876 - val_loss: 0.7284
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - accuracy: 0.8650 - loss: 0.4857 - val_accuracy: 0.8982 - val_loss: 0.4073
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - accuracy: 0.9473 - loss: 0.2394 - val_accuracy: 0.9216 - val_loss: 0.2917
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 8s 49ms/step - accuracy: 0.9705 - loss: 0.1416 - val_accuracy: 0.9260 - val_loss: 0.2515

PER-EPOCH RESULTS
epoch      loss    accuracy    val_loss   val_accuracy
    1    2.0889      0.3784      1.5990         0.5539
    2    1.0649      0.7053      0.7284         0.7876
    3    0.4857      0.8650      0.4073         0.8982
    4    0.2394      0.9473      0.2917         0.9216
    5    0.1416      0.9705      0.2515         0.9260

FINAL EPOCH
loss:  

---
# Task 4: Evaluating the tagger correctly &nbsp;&nbsp;*(4 marks)*

> **Watch out — this is the trap in this assignment.** Most positions in `y_test` are **padding**,
> not real words. An accuracy computed over the whole array is meaningless: it is dominated by
> padding, and depending on how you compute it you can get a number that is far too high *or* far
> too low. Neither tells you how good your tagger is.
>
> **Report accuracy over real tokens only** — every position where the true tag id is not the
> padding id. Compute it yourself; do not simply quote whatever `model.evaluate()` prints.

### Part A — masked accuracy

Write `masked_accuracy(model, X, y)` returning the fraction of **non-padding** positions where the
predicted tag id equals the true one. Also print the accuracy `model_lstm.evaluate(...)` gives, and
**state in one line whether the two agree and why**.

### Part B — per-tag accuracy

Report accuracy separately for each of the 12 tags, sorted worst first. This shows *which* parts of
speech are hard, which is far more useful than a single number.

### Part C — tag new sentences

Write `tag_sentence(model, sentence)` taking a plain string and returning `(word, predicted_tag)`
pairs. Run it on:

* `The quick brown fox jumps over the lazy dog .`
* `He will book a flight to New York tomorrow .`
* `They are reading a book about the stock market .`

### Output your code must print

```
Masked accuracy (real tokens only): 0.____
Keras evaluate() accuracy:          0.____
[one line explaining whether they agree and why]

PER-TAG ACCURACY
tag      n      accuracy
...

The/DET quick/ADJ brown/ADJ fox/NOUN ...
```

### Hints

- `model.predict(X)` returns `(n, MAX_LEN, n_tags+1)`. Use `.argmax(-1)` for predicted ids.
- Build the mask as `y_true != 0` and compare only those positions.
- Words missing from `word_index` must map to the unknown id, exactly as in Task 2.
- Sentences 2 and 3 are chosen deliberately: **"book"** is a verb in one and a noun in the other.
  Check whether your tagger gets both right.
- Sentence 1 is ordinary English, but the Treebank is **Wall Street Journal text**. Expect some odd
  tags, and think about why.

In [6]:
# ---------------- TASK 4 ----------------

# -----------------------------------
# TODO 17-19: Masked accuracy
# -----------------------------------

def masked_accuracy(model, X, y):
    """
    Accuracy over real tokens only.
    Padding positions (tag id 0) are ignored.
    """

    probabilities = model.predict(
        X,
        verbose=0
    )

    predictions = probabilities.argmax(
        axis=-1
    )

    mask = (
        y != 0
    )

    correct = (
        predictions == y
    ) & mask

    return (
        correct.sum()
        / mask.sum()
    )


# -----------------------------------
# TODO 20-22: Tag one sentence
# -----------------------------------

def tag_sentence(model, sentence):
    """
    Input:
        sentence: plain string

    Output:
        list of (original_word, predicted_tag)
    """

    original_words = (
        sentence.split()
    )

    word_ids = [
        word_index.get(
            word.lower(),
            1
        )
        for word in original_words
    ]

    X_new = pad_sequences(
        [word_ids],
        maxlen=MAX_LEN,
        padding="post",
        value=0
    )

    probabilities = model.predict(
        X_new,
        verbose=0
    )

    predicted_ids = probabilities[0].argmax(
        axis=-1
    )

    id_to_tag = {
        index: tag
        for tag, index in tag_index.items()
    }

    result = []

    for word, tag_id in zip(
        original_words,
        predicted_ids[:len(original_words)]
    ):

        predicted_tag = id_to_tag.get(
            int(tag_id),
            "X"
        )

        result.append(
            (word, predicted_tag)
        )

    return result


# -----------------------------------
# TODO 23: Overall evaluation
# -----------------------------------

masked_acc = masked_accuracy(
    model_lstm,
    X_test,
    y_test
)

evaluate_loss, evaluate_acc = (
    model_lstm.evaluate(
        X_test,
        y_test,
        verbose=0
    )
)

print(
    f"Masked accuracy (real tokens only): "
    f"{masked_acc:.4f}"
)

print(
    f"Keras evaluate() accuracy:          "
    f"{evaluate_acc:.4f}"
)

if abs(
    masked_acc - evaluate_acc
) < 0.001:

    print(
        "They agree because mask_zero=True "
        "causes padded positions to be ignored."
    )

else:

    print(
        "They differ because the two accuracy "
        "calculations handle padding differently."
    )


# -----------------------------------
# TODO 24: Per-tag accuracy
# -----------------------------------

test_probabilities = model_lstm.predict(
    X_test,
    verbose=0
)

test_predictions = (
    test_probabilities.argmax(
        axis=-1
    )
)

print("\nPER-TAG ACCURACY")
print(
    f"{'tag':<8}"
    f"{'n':>8}"
    f"{'accuracy':>12}"
)

tag_results = []

for tag, tag_id in tag_index.items():

    if tag_id == 0:
        continue

    mask = (
        y_test == tag_id
    )

    n = int(
        mask.sum()
    )

    if n == 0:
        continue

    correct = (
        (test_predictions == tag_id)
        & mask
    ).sum()

    accuracy = (
        correct / n
    )

    tag_results.append(
        (
            tag,
            n,
            accuracy
        )
    )

tag_results.sort(
    key=lambda item: item[2]
)

for tag, n, accuracy in tag_results:

    print(
        f"{tag:<8}"
        f"{n:>8}"
        f"{accuracy:>12.4f}"
    )


# -----------------------------------
# TODO 25: Three required sentences
# -----------------------------------

sentences = [
    "The quick brown fox jumps over the lazy dog .",
    "He will book a flight to New York tomorrow .",
    "They are reading a book about the stock market ."
]

print("\nTAGGED SENTENCES")

for sentence in sentences:

    tagged = tag_sentence(
        model_lstm,
        sentence
    )

    print(
        " ".join(
            f"{word}/{tag}"
            for word, tag in tagged
        )
    )

Masked accuracy (real tokens only): 0.9326
Keras evaluate() accuracy:          0.9326
They agree because mask_zero=True causes padded positions to be ignored.

PER-TAG ACCURACY
tag            n    accuracy
ADJ         1410      0.7319
ADV          594      0.7727
NUM         1208      0.7939
VERB        3034      0.9249
NOUN        6797      0.9360
DET         1955      0.9708
ADP         2288      0.9781
PRT          777      0.9897
X           1498      0.9973
PRON         407      0.9975
.           2576      0.9977
CONJ         499      0.9980

TAGGED SENTENCES
The/DET quick/ADJ brown/NOUN fox/NOUN jumps/VERB over/ADP the/DET lazy/NOUN dog/NOUN ./.
He/PRON will/VERB book/VERB a/DET flight/NOUN to/PRT New/ADJ York/NOUN tomorrow/NOUN ./.
They/PRON are/VERB reading/NOUN a/DET book/NOUN about/ADP the/DET stock/NOUN market/NOUN ./.


---
# Task 5: Bidirectional LSTM and comparison &nbsp;&nbsp;*(4 marks)*

### The idea

Your LSTM reads strictly **left to right**. When it tags a word it has seen everything before it
and **nothing after it**. But the disambiguating words are often on the right:

> *He will **book** a flight.* &nbsp;&nbsp;vs&nbsp;&nbsp; *She read a **book** about markets.*

A **bidirectional** RNN runs two LSTMs — one forward, one backward — and joins their hidden states
at each position:

$$h_t = \left[\overrightarrow{h_t}\; ;\; \overleftarrow{h_t}\right]$$

So every word is tagged using the whole sentence. Wrapping a layer in `Bidirectional(...)` is all
Keras needs; note that it **doubles** the output width per step.

> Bidirectionality is only possible because the **entire** sentence is available up front. You
> could not do this in a language model that must predict the next word — which is why Lab 6 did
> not use it.

### What your code must create

| Name | Is |
|---|---|
| `model_bilstm` | Task 3's model with the LSTM wrapped in `Bidirectional` |
| `history_bilstm` | its training history |

Train it **exactly** as you trained the LSTM — same epochs, batch size and split — or the
comparison is not fair.

### Output your code must print

```
MODEL COMPARISON (masked accuracy on the test set)
  LSTM      params ______   accuracy 0.____
  BiLSTM    params ______   accuracy 0.____
  improvement: +_.__ percentage points

[the three sentences tagged by the BiLSTM]
[any words where the two models disagree]
```

### Then answer

**Q2.** How many percentage points did bidirectionality gain, and how many extra parameters did it
cost? Was it worth it?

**Q3.** List every word where the two models disagree. Pick one and explain, using right-hand
context, why seeing the rest of the sentence could change the decision. *(Look closely at the
proper noun in sentence 2.)* Also say whether **"book"** needed bidirectionality at all, or whether
left context was already enough.

**Q4.** From your per-tag table, which tags were hardest? Sentence 1 is plain English yet gets
several tags wrong — what does that tell you about the data the model was trained on?

### Hints

- Only one line of the model changes. Training, evaluation and `masked_accuracy` are reused as is.
- Call `keras.utils.set_random_seed(42)` again before building the second model, so the comparison
  is not distorted by different random initialisation.
- `model.count_params()` gives the parameter count.
- To find disagreements, tag the same sentence with both models and compare position by position.

In [7]:
# ---------------- TASK 5 ----------------

# -----------------------------------
# TODO 26: Build BiLSTM
# -----------------------------------

keras.utils.set_random_seed(42)

model_bilstm = Sequential([
    Input(
        shape=(MAX_LEN,)
    ),

    Embedding(
        input_dim=len(words) + 2,
        output_dim=EMBED_DIM,
        mask_zero=True
    ),

    Bidirectional(
        LSTM(
            RNN_UNITS,
            return_sequences=True
        )
    ),

    TimeDistributed(
        Dense(
            len(tags) + 1,
            activation="softmax"
        )
    )
])

# -----------------------------------
# TODO 27: Compile and train
# -----------------------------------

model_bilstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nBIDIRECTIONAL LSTM")
print("=" * 60)

model_bilstm.summary()

history_bilstm = model_bilstm.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# -----------------------------------
# TODO 28: Compare models
# -----------------------------------

lstm_accuracy = masked_accuracy(
    model_lstm,
    X_test,
    y_test
)

bilstm_accuracy = masked_accuracy(
    model_bilstm,
    X_test,
    y_test
)

lstm_params = model_lstm.count_params()
bilstm_params = model_bilstm.count_params()

improvement_pp = (
    bilstm_accuracy - lstm_accuracy
) * 100

extra_params = (
    bilstm_params - lstm_params
)

print(
    "\nMODEL COMPARISON "
    "(masked accuracy on the test set)"
)

print(
    f"  LSTM      params "
    f"{lstm_params:,}   "
    f"accuracy {lstm_accuracy:.4f}"
)

print(
    f"  BiLSTM    params "
    f"{bilstm_params:,}   "
    f"accuracy {bilstm_accuracy:.4f}"
)

print(
    f"  improvement: "
    f"{improvement_pp:+.2f} "
    f"percentage points"
)

print(
    f"  extra parameters: "
    f"{extra_params:,}"
)

# -----------------------------------
# TODO 29: Tag three sentences
# -----------------------------------

print("\nBiLSTM TAGGED SENTENCES")

for sentence in sentences:

    tagged = tag_sentence(
        model_bilstm,
        sentence
    )

    print(
        " ".join(
            f"{word}/{tag}"
            for word, tag in tagged
        )
    )

# -----------------------------------
# TODO 30: Disagreements
# -----------------------------------

print("\nMODEL DISAGREEMENTS")

for sentence in sentences:

    lstm_result = tag_sentence(
        model_lstm,
        sentence
    )

    bilstm_result = tag_sentence(
        model_bilstm,
        sentence
    )

    print(
        f"\nSentence: {sentence}"
    )

    found = False

    for (
        (word_lstm, tag_lstm),
        (word_bilstm, tag_bilstm)
    ) in zip(
        lstm_result,
        bilstm_result
    ):

        if tag_lstm != tag_bilstm:

            found = True

            print(
                f"{word_lstm}: "
                f"LSTM={tag_lstm} -> "
                f"BiLSTM={tag_bilstm}"
            )

    if not found:
        print("No disagreements")


BIDIRECTIONAL LSTM


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 50, 64)         │       728,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 50, 128)        │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 50, 13)         │         1,677 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 796,621 (3.04 MB)

 Trainable params: 796,621 (3.04 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 14s 116ms/step - accuracy: 0.3730 - loss: 2.0194 - val_accuracy: 0.5756 - val_loss: 1.4351
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 93ms/step - accuracy: 0.7464 - loss: 0.8727 - val_accuracy: 0.8599 - val_loss: 0.5258
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 12s 113ms/step - accuracy: 0.9115 - loss: 0.3240 - val_accuracy: 0.9250 - val_loss: 0.2787
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.9632 - loss: 0.1560 - val_accuracy: 0.9391 - val_loss: 0.2131
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 90ms/step - accuracy: 0.9775 - loss: 0.0996 - val_accuracy: 0.9447 - val_loss: 0.1903

MODEL COMPARISON (masked accuracy on the test set)
  LSTM      params 762,765   accuracy 0.9326
  BiLSTM    params 796,621   accuracy 0.9474
  improvement: +1.48 percentage points
  extra parameters: 33,856

BiLSTM TAGGED SENTENCES
The/DET quick/ADJ brown/NOUN fox/VERB jumps/NOUN over/ADP the/DET lazy/NOUN dog/NOUN ./.
He/PRON will/VERB book/VERB a/DET flight/N

### Your answers

**Q1.** Where the LSTM's improvement was largest, and whether that matches the vanishing-gradient explanation:

The LSTM's improvement was largest on long sentences. It improved accuracy by **11.9 percentage points** on long sentences, compared with **8.7 percentage points** on short sentences. This matches the vanishing-gradient explanation because longer sequences require information and gradients to pass through more recurrent steps, making it harder for a SimpleRNN to preserve long-range information. The LSTM's gated cell state helps preserve this information over longer sequences.

---

**Q2.** Gain from bidirectionality, and its cost in parameters:

The BiLSTM improved masked test accuracy from **0.9326 to 0.9474**, giving a gain of **1.48 percentage points**. The LSTM has **762,765 parameters**, while the BiLSTM has **796,621 parameters**, so bidirectionality costs **33,856 additional parameters**. The improvement is worthwhile because the BiLSTM can use both left-hand and right-hand context when assigning a POS tag.

---

**Q3.** Where the two models disagree, and what right-hand context explains it:

The two models disagree on **fox, jumps, New, and reading**. Specifically, the LSTM predicts `fox/NOUN`, `jumps/VERB`, `New/ADJ`, and `reading/NOUN`, while the BiLSTM predicts `fox/VERB`, `jumps/NOUN`, `New/NOUN`, and `reading/VERB`. The BiLSTM can use right-hand context because it reads the sentence in both directions. For example, seeing the word **York** after **New** gives useful context for recognizing the phrase **New York** as a proper name. For **book**, the left context is already enough: “will book” strongly indicates a verb, while “a book” indicates a noun, so bidirectionality is not necessarily needed for that word.

---

**Q4.** The hardest tags, and what sentence 1 reveals about the training data:

The hardest tags were **ADJ (0.7319)**, **ADV (0.7727)**, and **NUM (0.7939)** because they had the lowest per-tag accuracies. Sentence 1 also shows that the model was trained on **Wall Street Journal text** from the Penn Treebank rather than general everyday English. Therefore, the model has learned patterns from a specific news-writing domain, which can cause unexpected predictions on an ordinary sentence such as “The quick brown fox jumps over the lazy dog.”

---
## Submission checklist

- [ ] `Runtime → Restart and run all` completes without errors
- [ ] **Task 1** — the three-way comparison table against the SimpleRNN baseline
- [ ] **Task 2** — statistics printed, and one example shown in both word and id form
- [ ] **Task 3** — `model_lstm.summary()` and the per-epoch table are visible
- [ ] **Task 4** — masked accuracy, the comparison with `evaluate()`, the per-tag table, and the
      three tagged sentences
- [ ] **Task 5** — both models compared on parameters and accuracy, BiLSTM tagging shown,
      disagreements listed
- [ ] All four questions answered
- [ ] Submit the `.ipynb` with all outputs saved

### Common mistakes

1. Changing something other than the recurrent layer in Task 1, making the comparison meaningless.
2. Using id 0 for a real word or a real tag — 0 must mean padding and nothing else.
3. Padding at the front (`padding='pre'`) for tagging, which misaligns words and tags.
4. Forgetting `return_sequences=True`, so the LSTM emits one vector instead of one per word.
5. Reporting raw accuracy over the whole padded array instead of over real tokens only.
6. Training the two taggers for different numbers of epochs.
7. Looking words up without lowercasing them, when the vocabulary was built lowercased.